# Step 13 — CTO recommendation validation

This notebook validates the one-page recommendation requested by the assignment. It checks that the document makes a clear ETL/ELT decision, compares all required dimensions, uses pipeline evidence, and ends with three risks.

In [1]:
# Read the recommendation from the repository without changing it.
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'RECOMMENDATION_TO_CTO.md').exists():
    REPO_ROOT = Path.cwd().parent
recommendation_path = REPO_ROOT / 'RECOMMENDATION_TO_CTO.md'
recommendation = recommendation_path.read_text()
print(f'Reading: {recommendation_path}')
print(f'Word count: {len(recommendation.split())}')

Reading: /Users/kumarsamanvaya/Documents/IITR - FDE Course/Data Pipepline Assignment/_Samy_IIT_Roorkie_CEC_Futurenece_Data_Pipeline_to_ML_Model/RECOMMENDATION_TO_CTO.md
Word count: 748


In [2]:
# Confirm that every required comparison appears in the one-page document.
required_topics = [
    'Auditability',
    'Data-quality guarantees',
    'Ease of changing rules',
    'Performance at 1,000× volume',
    'Cost',
    'Operational risks',
]
missing_topics = [topic for topic in required_topics if topic.lower() not in recommendation.lower()]
assert not missing_topics, f'Missing comparison sections: {missing_topics}'
assert 'adopt **elt' in recommendation.lower(), 'The recommendation must make a clear ELT decision.'
assert 500 <= len(recommendation.split()) <= 1000, 'Keep the recommendation close to one page.'
print('Decision and comparison checks passed.')

Decision and comparison checks passed.


In [3]:
# Check that the recommendation ends with exactly three numbered risks.
risk_heading = '## Three biggest risks of this recommendation'
assert risk_heading in recommendation
risk_section = recommendation.split(risk_heading, 1)[1]
risk_lines = [line for line in risk_section.splitlines() if line.startswith(('1. ', '2. ', '3. '))]
assert len(risk_lines) == 3, f'Expected exactly three final risks, found {len(risk_lines)}'
print('Final-risk check passed: exactly three risks are documented.')
print('\n'.join(risk_lines))

Final-risk check passed: exactly three risks are documented.
1. **Bad data reaches a mart** if a model forgets to filter `dq_status`; mitigate with certified finance views, SQL tests, and restricted write access.
2. **Warehouse cost and latency grow** at high volume; mitigate with partitioning, incremental models, retention controls, and workload monitoring.
3. **Raw PII is overexposed** because ELT retains complete source data; mitigate with role-based access, masking in staging/marts, secret management, and audit logs.


In [4]:
# Display the complete recommendation so it can be reviewed in the notebook.
print(recommendation)
print('\nStep 13 validation passed.')

# Recommendation to Sameer, CTO: adopt ELT with a controlled finance contract

## Decision

NammaMart should adopt **ELT as the primary warehouse architecture**, with a validated finance-facing contract that publishes only approved rows and metrics. ELT is the better system of record because it preserves every source row, makes transformations replayable, and keeps rule changes in version-controlled SQL. ETL should remain available as a strict publishing step for finance-critical tables, where rejected records must never enter revenue reporting.

## Evidence from the completed work

The September test run loaded 1,850 clean ETL fact rows. The ELT run retained 2,412 raw order rows and produced 2,316 passing staging rows. This demonstrates the central trade-off: ETL gives a smaller, strongly filtered finance dataset, while ELT preserves more evidence for audit and investigation. The pipelines also detected 27 failed data-quality checks, and the incremental test handled 150 new orders and